# Airflow Variable Precedence — Behaviour Test (`nx1_` S3 globals)

Verifies `migrator_utils/migrations/shared.py::get_config._var` on a live tenant: every
resolution tier, in every combination of set / empty / absent, for both portal-triggered and
hand-launched runs.

### The resolution under test

| # | Source | Who reaches it | Matches on |
|---|--------|----------------|------------|
| 1 | `<key>__<run_id>` | portal (only it writes these) | **presence** |
| 2 | `nx1_<key>` | **portal-triggered runs only** | non-empty |
| 3 | `<key>` plain Variable | everyone | non-empty |
| 4 | `<ENV_VAR>` from `env.shared` / `env.<dag_stem>` | everyone | non-empty |
| 5 | hardcoded default | everyone | — |

Tier 2 is gated on `dag_run.conf['triggered_by'] == 'portal'`. Tier 1 matching on presence
rather than truthiness is what lets a run clear `migration_email_recipients` to mean "send no
report" instead of inheriting the tenant mailing list.

### How it works

A throwaway DAG (`nx1_var_probe`) is uploaded to the tenant's DAG folder. Its single task calls
the **deployed** `get_config()` and reports what every key resolved to. The notebook sets up
Variables so that each config key encodes one scenario, triggers the probe several times with
different `conf`, and asserts the tier that won.

Each scenario gets its own carrier key, so one DAG run evaluates the whole matrix at once.
Runs are cheap — no Spark, no data.

| Run | `conf.triggered_by` | Expectation column |
|---|---|---|
| `portal` | `portal` | portal |
| `portal_mixedcase` | `"  PoRtAl  "` | portal (marker is stripped + lowercased) |
| `manual` | *(absent)* | manual |
| `other_marker` | `manual` | manual |
| `owner_fallback` | `portal` | dedicated check for `get_config`'s owner chain |
| `baseline_pre` / `baseline_post` | `portal` | no test Variables at all — regression + leak check |

### ⚠️ Read before running

**This notebook temporarily rewrites tenant-wide Airflow Variables** — plain keys such as
`auth_method`, `cluster_type`, `migration_tracking_database`, and the `nx1_s3_*` globals the
portal's Infrastructure Config page owns. A real migration DAG running in that window would
resolve the probe's values.

- The preflight cell refuses to continue while any migration DAG run is active.
- Every touched key's prior value is saved before the first write and restored by the cleanup
  cell, including deleting keys that did not exist before.
- The backup is also written to `~/nx1_var_probe_variable_backup.json` (mode 600) so a dead
  kernel is recoverable — the last cell restores from it. **It contains real secret values**;
  cleanup deletes it.

Run cells top to bottom. If you stop early, run the cleanup cell.

### Prerequisites

- Airflow webserver reachable from JH, with credentials allowed to write Variables and trigger
  DAGs (in-cluster URL bypasses Keycloak).
- JH Spark session with write access to the DAG bucket (used to place and remove the probe DAG).
- `DEPLOY_S3_BUCKET` / `DEPLOY_DAGS_PREFIX` matching the tenant's `env.shared`.

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────────
import os

# --- Airflow REST API ---
# In-cluster URL bypasses Keycloak (which only fronts the public Ingress).
# Example: "http://airflow-api-server.<tenant-namespace>.svc.cluster.local:8080"
AIRFLOW_BASE_URL = os.environ.get("AIRFLOW_BASE_URL", "")
AIRFLOW_USERNAME = os.environ.get("AIRFLOW_USERNAME", "")
AIRFLOW_PASSWORD = os.environ.get("AIRFLOW_PASSWORD", "")

# --- DAG folder on S3 (must match env.shared's DEPLOY_* values) ---
# The API writes the same prefix from _upload_migration_dags at startup.
DAGS_S3_BUCKET = os.environ.get("DEPLOY_S3_BUCKET", "")
DAGS_S3_PREFIX = os.environ.get("DEPLOY_DAGS_PREFIX", "")   # e.g. "airflow/es-tenant-2/dags/"

# --- Probe DAG ---
PROBE_DAG_ID = "nx1_var_probe"
PROBE_S3_KEY = f"{DAGS_S3_PREFIX}{PROBE_DAG_ID}.py"
PROBE_S3_URI = f"s3a://{DAGS_S3_BUCKET}/{PROBE_S3_KEY}"

# Optional: also load env.<stem> in the baseline runs, to reproduce exactly what one
# deployed DAG resolves. deploy.py names these env.<dag_stem>_<suffix> — e.g.
# "migration_dag_mapr_to_s3_aleks". Leave blank to use env.shared only.
BASELINE_ENV_STEM = os.environ.get("PROBE_ENV_STEM", "")

# DAG ids that must not be running while the probe rewrites shared Variables.
GUARDED_DAG_PREFIXES = ("source_to_s3_migration", "iceberg_migration",
                        "folder_only_data_copy", "parquet_hms_registration")

VARIABLE_BACKUP_PATH = os.path.expanduser("~/nx1_var_probe_variable_backup.json")

# Print real credential values instead of sha256[:8] fingerprints.
#
# The probe cannot hand a value to this notebook without putting it somewhere
# Airflow keeps, so turning this on means the plaintext lands in:
#   * the XCom row for each probe run  — purge_probe_xcoms() below deletes these
#   * this notebook's saved output     — clear the cells before committing
# The task log is deliberately skipped when this is on. Leave it off unless you
# are actively chasing a credential mismatch.
REVEAL_SECRETS = False

_missing = [name for name, value in [
    ("AIRFLOW_BASE_URL", AIRFLOW_BASE_URL),
    ("AIRFLOW_USERNAME", AIRFLOW_USERNAME),
    ("AIRFLOW_PASSWORD", AIRFLOW_PASSWORD),
    ("DEPLOY_S3_BUCKET", DAGS_S3_BUCKET),
    ("DEPLOY_DAGS_PREFIX", DAGS_S3_PREFIX),
] if not value]

print("Configuration loaded.")
print(f"  Airflow URL     : {AIRFLOW_BASE_URL or '(unset)'}")
print(f"  Airflow user    : {AIRFLOW_USERNAME or '(unset)'}")
print(f"  Probe DAG       : {PROBE_DAG_ID}")
print(f"  Probe DAG file  : {PROBE_S3_URI if DAGS_S3_BUCKET else '(unset)'}")
print("  Baseline env    : env.shared" + (f" + env.{BASELINE_ENV_STEM}" if BASELINE_ENV_STEM else ""))
print(f"  Variable backup : {VARIABLE_BACKUP_PATH}")
if _missing:
    print(f"\n  MISSING env vars: {', '.join(_missing)} — set them before continuing.")

In [ ]:
# ── Airflow REST helpers ──────────────────────────────────────────────────────────
# Version detection mirrors api/api/clients/airflow_client.py: /api/v2/version 404
# means Airflow 2 (/api/v1 + BasicAuth); anything else means Airflow 3 (/api/v2 +
# JWT from /auth/token). A 401 on that probe still means v2 exists.
import hashlib
import json
import re
import time
from datetime import datetime, timedelta, timezone

import requests

_STATE = {"prefix": None, "major": None, "token": None, "logical_seq": 0}


def _api_prefix():
    if _STATE["prefix"]:
        return _STATE["prefix"]
    url = f"{AIRFLOW_BASE_URL.rstrip('/')}/api/v2/version"
    try:
        status = requests.get(url, timeout=10).status_code
    except Exception as exc:
        raise RuntimeError(f"Cannot reach {url}: {exc}") from exc
    _STATE["major"] = 2 if status == 404 else 3
    _STATE["prefix"] = "/api/v1" if status == 404 else "/api/v2"
    print(f"Airflow {_STATE['major']} detected ({url} -> {status}); using {_STATE['prefix']}")
    return _STATE["prefix"]


def _token():
    if _STATE["token"]:
        return _STATE["token"]
    url = f"{AIRFLOW_BASE_URL.rstrip('/')}/auth/token"
    resp = requests.post(url, json={"username": AIRFLOW_USERNAME,
                                    "password": AIRFLOW_PASSWORD}, timeout=30)
    if not resp.ok:
        raise RuntimeError(f"POST {url} -> {resp.status_code}: {resp.text[:300]}")
    _STATE["token"] = resp.json()["access_token"]
    return _STATE["token"]


def af(method, path, allow=(), raw=False, _retry=True, **kwargs):
    """Call the Airflow API. Status codes in `allow` return None instead of raising."""
    prefix = _api_prefix()
    url = f"{AIRFLOW_BASE_URL.rstrip('/')}{prefix}{path}"
    if _STATE["major"] == 2:
        kwargs["auth"] = (AIRFLOW_USERNAME, AIRFLOW_PASSWORD)
    else:
        kwargs.setdefault("headers", {})["Authorization"] = f"Bearer {_token()}"
    resp = requests.request(method, url, timeout=60, **kwargs)
    if resp.status_code == 401 and _retry and _STATE["major"] == 3:
        _STATE["token"] = None
        kwargs.get("headers", {}).pop("Authorization", None)
        return af(method, path, allow=allow, raw=raw, _retry=False, **kwargs)
    if resp.status_code in allow:
        return None
    if not resp.ok:
        raise RuntimeError(f"{method} {url} -> {resp.status_code}: {resp.text[:400]}")
    if raw:
        return resp.text
    return resp.json() if resp.content else {}


# ── Variables ────────────────────────────────────────────────────────────
def var_get(key):
    result = af("GET", f"/variables/{key}", allow=(404,))
    return None if result is None else result.get("value")


def var_set(key, value):
    body = {"key": key, "value": value}
    if af("POST", "/variables", json=body, allow=(409,)) is None:
        af("PATCH", f"/variables/{key}", json=body)


def var_delete(key):
    af("DELETE", f"/variables/{key}", allow=(404,))


# ── DAG runs ─────────────────────────────────────────────────────────────
PROBE_RUN_IDS: list[str] = []


def trigger(run_id, conf, dag_id=None):
    """Trigger one probe run. Logical dates are spaced apart because Airflow 3
    enforces uniqueness per (dag_id, logical_date) and we fire several at once."""
    _STATE["logical_seq"] += 1
    stamp = datetime.now(timezone.utc) + timedelta(seconds=_STATE["logical_seq"])
    body = {
        "dag_run_id": run_id,
        "conf": conf,
        "logical_date": stamp.strftime("%Y-%m-%dT%H:%M:%S+00:00"),
    }
    path = f"/dags/{dag_id or PROBE_DAG_ID}/dagRuns"
    PROBE_RUN_IDS.append(run_id)
    try:
        return af("POST", path, json=body)
    except RuntimeError as exc:
        if "logical_date" not in str(exc):
            raise
        body.pop("logical_date")
        return af("POST", path, json=body)


def wait_for(run_ids, timeout=900, interval=10):
    terminal = {"success", "failed"}
    pending, states = list(run_ids), {}
    deadline = time.time() + timeout
    while pending and time.time() < deadline:
        for run_id in list(pending):
            states[run_id] = af("GET", f"/dags/{PROBE_DAG_ID}/dagRuns/{run_id}").get("state")
            if states[run_id] in terminal:
                pending.remove(run_id)
                print(f"  {run_id}: {states[run_id]}")
        if pending:
            time.sleep(interval)
    for run_id in pending:
        print(f"  {run_id}: TIMED OUT (last state {states.get(run_id)})")
    return states


# ── Probe payload retrieval ──────────────────────────────────────────────
PAYLOAD_BEGIN = "NX1_VAR_PROBE_PAYLOAD_BEGIN"
PAYLOAD_END = "NX1_VAR_PROBE_PAYLOAD_END"


def _loads(raw):
    """XCom hands back a JSON-encoded string; some API versions double-encode it."""
    value = raw
    for _ in range(3):
        if isinstance(value, dict):
            return value
        if not isinstance(value, str):
            break
        try:
            value = json.loads(value)
        except (TypeError, ValueError):
            break
    raise RuntimeError(f"Could not parse probe payload: {str(raw)[:300]}")


def _scrape_log(run_id, task_id):
    text = af("GET", f"/dags/{PROBE_DAG_ID}/dagRuns/{run_id}/taskInstances/{task_id}/logs/1",
              raw=True, allow=(404,))
    if not text:
        return None
    match = re.search(f"{PAYLOAD_BEGIN}(.*?){PAYLOAD_END}", text, re.S)
    if not match:
        return None
    body = match.group(1).replace("\\n", "\n")
    start, end = body.find("{"), body.rfind("}")
    return body[start:end + 1] if start >= 0 and end > start else None


def fetch_payload(run_id, task_id="probe"):
    path = (f"/dags/{PROBE_DAG_ID}/dagRuns/{run_id}/taskInstances/{task_id}"
            f"/xcomEntries/return_value")
    raw = None
    for params in ({}, {"deserialize": "true"}):
        result = af("GET", path, params=params, allow=(400, 404, 500))
        if result and result.get("value"):
            raw = result["value"]
            break
    if raw is None:
        raw = _scrape_log(run_id, task_id)
    return _loads(raw)


def purge_probe_xcoms():
    """Delete every probe DAG run, and with it the XCom holding its payload.

    Only matters when REVEAL_SECRETS was on: that is the one path where a real
    credential is written into Airflow's metadata DB. Deleting the run is the
    coarse-but-certain way to remove its XCom through the public API.
    """
    purged, failed = [], []
    for run_id in PROBE_RUN_IDS:
        try:
            af("DELETE", f"/dags/{PROBE_DAG_ID}/dagRuns/{run_id}", allow=(404,))
            purged.append(run_id)
        except RuntimeError as exc:
            failed.append((run_id, str(exc)))
    return purged, failed


# ── Secret handling ──────────────────────────────────────────────────────
# get_config returns live credentials, so the probe fingerprints anything that
# looks like one instead of putting it in XCom. Both sides must agree on the rule.
SECRET_HINTS = ("secret", "password", "access_key", "keytab")

# What Airflow's REST API returns instead of a credential-shaped Variable's value.
# Reading one and writing it back would replace the credential with this.
MASK = "***"


def is_secret(key):
    return any(hint in key.lower() for hint in SECRET_HINTS)


def fingerprint(value):
    text = "" if value is None else str(value)
    return {"redacted": True, "len": len(text),
            "sha256_8": hashlib.sha256(text.encode("utf-8")).hexdigest()[:8]}


print("Helpers ready.")

In [ ]:
# ── S3 helpers (Spark's Hadoop FS — same credentials the other notebooks use) ────
from py4j.java_gateway import java_import
from pyspark.sql import SparkSession

try:
    _ = spark
    print(f"Using existing Spark session (version {spark.version})")
except NameError:
    spark = SparkSession.builder.appName("nx1-var-probe").getOrCreate()
    print(f"Created Spark session (version {spark.version})")

spark.sparkContext.setLogLevel("WARN")
java_import(spark._jvm, "org.apache.hadoop.fs.*")

_FS = spark._jvm.org.apache.hadoop.fs.FileSystem
_Path = spark._jvm.org.apache.hadoop.fs.Path
_URI = spark._jvm.java.net.URI


def _fs(uri):
    return _FS.get(_URI(uri), spark._jsc.hadoopConfiguration())


def s3_put_text(uri, text):
    stream = _fs(uri).create(_Path(uri), True)
    try:
        stream.write(bytearray(text.encode("utf-8")))
    finally:
        stream.close()
    return len(text)


def s3_delete(uri):
    fs, path = _fs(uri), _Path(uri)
    if fs.exists(path):
        fs.delete(path, True)
        return True
    return False


def s3_read_text(uri):
    stream = _fs(uri).open(_Path(uri))
    try:
        reader = spark._jvm.java.io.BufferedReader(
            spark._jvm.java.io.InputStreamReader(stream, "UTF-8")
        )
        lines, line = [], reader.readLine()
        while line is not None:
            lines.append(line)
            line = reader.readLine()
        reader.close()
    finally:
        stream.close()
    return "\n".join(lines)


def s3_ls(uri):
    fs, path = _fs(uri), _Path(uri)
    if not fs.exists(path):
        return []
    out = []
    for status in fs.listStatus(path):
        out.append({
            "path": status.getPath().toString(),
            "bytes": status.getLen(),
            "modified": datetime.fromtimestamp(
                status.getModificationTime() / 1000, timezone.utc
            ).strftime("%Y-%m-%d %H:%M:%S UTC"),
            "dir": status.isDirectory(),
        })
    return sorted(out, key=lambda row: row["path"])


print(f"\nDAG folder — s3a://{DAGS_S3_BUCKET}/{DAGS_S3_PREFIX}")
for row in s3_ls(f"s3a://{DAGS_S3_BUCKET}/{DAGS_S3_PREFIX}"):
    kind = "dir " if row["dir"] else "file"
    print(f"  {kind} {row['bytes']:>9}  {row['modified']}  {row['path'].rsplit('/', 1)[-1]}")

_shared_uri = f"s3a://{DAGS_S3_BUCKET}/{DAGS_S3_PREFIX}migrator_utils/migrations/shared.py"
print("\nDeployed shared.py — the copy every DAG imports:")
for row in s3_ls(_shared_uri.rsplit("/", 1)[0]):
    if row["path"].endswith("shared.py"):
        print(f"  {row['bytes']} bytes, last written {row['modified']}")

---
## Step 1 — The probe DAG

`nx1_var_probe` imports `get_config` from the **deployed** `migrator_utils.migrations.shared`,
so it exercises whichever copy is live in the DAG folder. Two things write that same S3 key —
`deploy.py` from nx1-data-migrator and `_upload_migration_dags` in the API's startup — so the
probe also reports which one it found.

It mirrors a real DAG's parse-time `load_dotenv(env.shared)` so tier 4 is populated the same
way a production run would see it.

**`conf` keys, all optional:**

| Key | Effect |
|---|---|
| `triggered_by` | `portal` unlocks the `nx1_` tier — the same marker the API sets |
| `env_overlay` | `{ENV_VAR: value \| null}` applied around `get_config()`, then restored exactly. `null` unsets |
| `env_stem` | also load `env.<stem>` from the DAG folder, to reproduce one DAG's env |
| `report_env` | env vars to echo back, for diagnosing a failed expectation |
| `dag_owner` | feeds `get_config`'s owner fallback chain |

The env overlay is restored in a `finally` block: worker processes are reused across DAG runs,
so a leaked override would change what the next task on that worker resolves.

Values whose key looks like a credential are replaced by a `sha256[:8]` + length fingerprint
before leaving the task — the tenant's real `S3_SECRET_KEY` would otherwise land in XCom and
the task log. The notebook fingerprints its expected values the same way and compares those.

In [ ]:
# ── Probe DAG source ─────────────────────────────────────────────────────────────
PROBE_DAG_SOURCE = r'''
"""nx1_var_probe — Airflow Variable precedence probe (TEST DAG).

Uploaded by notebooks/test_airflow_variable_precedence.ipynb and removed again by its
cleanup cell. Runs no Spark and touches no data: the single task calls the deployed
migrator_utils.migrations.shared.get_config() and reports what every key resolved to,
so the notebook can assert which tier won.
"""

import hashlib
import json
import logging
import os
from contextlib import contextmanager
from datetime import datetime
from pathlib import Path
from types import CodeType

from airflow import DAG
from airflow.decorators import task
from dotenv import load_dotenv

logger = logging.getLogger(__name__)

PAYLOAD_BEGIN = "NX1_VAR_PROBE_PAYLOAD_BEGIN"
PAYLOAD_END = "NX1_VAR_PROBE_PAYLOAD_END"

_dag_dir = Path(__file__).resolve().parent
_config_dir = _dag_dir / "migrator_utils" / "migration_configs"

# Every real migration DAG does this at parse time. Without it tier 4 would test as
# empty everywhere and the env-file scenarios would be meaningless.
if _config_dir.is_dir():
    load_dotenv(_config_dir / "env.shared")

# get_config returns live credentials. Anything matching these leaves the task as a
# fingerprint only — the notebook compares fingerprints of its expected values.
_SECRET_HINTS = ("secret", "password", "access_key", "keytab")


def _is_secret(key):
    return any(hint in key.lower() for hint in _SECRET_HINTS)


def _fingerprint(value):
    text = "" if value is None else str(value)
    return {"redacted": True, "len": len(text),
            "sha256_8": hashlib.sha256(text.encode("utf-8")).hexdigest()[:8]}


def _redact(mapping, reveal=False):
    if reveal:
        return dict(mapping)
    return {key: _fingerprint(value) if _is_secret(key) else value
            for key, value in mapping.items()}


def _current_context():
    """get_current_context moved to airflow.sdk in Airflow 3 — try both.

    get_config reaches for the airflow.operators.python path only. If that import
    fails, its run_id is None and tiers 1 and 2 silently stop resolving, so the
    module that worked is reported back.
    """
    for module, attr in (("airflow.sdk", "get_current_context"),
                         ("airflow.operators.python", "get_current_context")):
        try:
            mod = __import__(module, fromlist=[attr])
            return mod, getattr(mod, attr)(), module
        except Exception:
            continue
    return None, {}, None


@contextmanager
def _env_overlay(overlay):
    """Apply {ENV_VAR: value|None} to os.environ, then put it back exactly.

    Workers are reused across DAG runs, so an override leaked from here would
    change what the next task on this worker resolves.
    """
    saved = {key: os.environ.get(key) for key in overlay}
    try:
        for key, value in overlay.items():
            if value is None:
                os.environ.pop(key, None)
            else:
                os.environ[key] = str(value)
        yield
    finally:
        for key, value in saved.items():
            if value is None:
                os.environ.pop(key, None)
            else:
                os.environ[key] = value


def _tier_snapshot(tier_report, run_id, portal_run, reveal):
    """Report what each tier actually holds for the given base keys.

    Deliberately does not work out which tier wins — that is _var's job, and a
    second implementation of the rule would just agree with itself. The caller
    lines these up against the resolved value instead, which also makes it
    visible when two tiers hold the same value and the source is ambiguous.
    """
    from airflow.models import Variable

    out = {}
    for base_key, env_var in (tier_report or {}).items():
        candidates = [
            ("run_scoped", f"{base_key}__{run_id}" if run_id else None),
            ("nx1", f"nx1_{base_key}"),
            ("plain", base_key),
        ]
        entry = {}
        for tier, name in candidates:
            if name is None:
                entry[tier] = {"source": None, "present": False, "value": None}
                continue
            try:
                value = Variable.get(name, default_var=None)
            except Exception as exc:
                entry[tier] = {"source": name, "error": f"{type(exc).__name__}: {exc}"}
                continue
            entry[tier] = {
                "source": name,
                "present": value is not None,
                "value": value if reveal or not _is_secret(base_key) or value is None
                         else _fingerprint(value),
            }
        env_value = os.getenv(env_var) if env_var else None
        entry["env"] = {
            "source": env_var,
            "present": env_value is not None,
            "value": env_value if reveal or not _is_secret(base_key) or env_value is None
                     else _fingerprint(env_value),
        }
        # The nx1_ tier is only consulted for portal-triggered runs, so a value
        # sitting there is not necessarily in play.
        entry["nx1"]["eligible"] = portal_run
        out[base_key] = entry
    return out


def _airflow_version():
    try:
        from airflow import __version__
        return __version__
    except Exception:
        return None


def _loaded_consts(code, depth=0):
    """Every string constant reachable from a loaded code object."""
    for const in code.co_consts:
        if isinstance(const, str):
            yield const
        elif isinstance(const, CodeType) and depth < 4:
            yield from _loaded_consts(const, depth + 1)


def _diagnostics():
    """Identify the deployed shared.py and whether it carries the new tiers.

    Marker checks rather than an md5 comparison: they keep working across edits
    unrelated to _var. The md5 is reported for identification only.

    The has_* markers read the file on disk, but get_config runs from whatever
    sys.modules already holds. A worker that imported shared.py before the deploy
    keeps serving the old code, so loaded_code_has_nx1_tier inspects the live code
    object's constants instead — that is the one that cannot lie.
    """
    from migrator_utils.migrations import shared as shared_mod

    path = Path(shared_mod.__file__)
    try:
        source = path.read_text(encoding="utf-8")
    except Exception as exc:
        logger.warning(f"could not read {path}: {exc}")
        source = ""

    var_src = ""
    if "def _var(" in source:
        var_src = source.split("def _var(", 1)[1].split("\n    dag_owner", 1)[0]

    try:
        from airflow.operators.python import get_current_context  # noqa: F401
        legacy_import = True
    except Exception:
        legacy_import = False

    consts = set(_loaded_consts(shared_mod.get_config.__code__))

    return {
        "shared_py_path": str(path),
        "shared_py_md5": hashlib.md5(source.encode("utf-8")).hexdigest() if source else None,
        "shared_py_bytes": len(source),
        "portal_trigger_const": getattr(shared_mod, "PORTAL_TRIGGER", None),
        "has_nx1_tier": "nx1_" in var_src,
        "loaded_code_has_nx1_tier": "nx1_" in consts,
        "tier1_on_presence": "is not None" in var_src,
        "swallows_lookup_errors": "except Exception" in var_src,
        "legacy_get_current_context_import": legacy_import,
        "env_shared_dir_present": _config_dir.is_dir(),
        "airflow_version": _airflow_version(),
    }


@task
def probe():
    _, ctx, ctx_module = _current_context()
    dag_run = ctx.get("dag_run")
    conf = dict(getattr(dag_run, "conf", None) or {})

    overlay = dict(conf.get("env_overlay") or {})
    env_stem = conf.get("env_stem")
    report_env = list(conf.get("report_env") or [])
    reveal = bool(conf.get("reveal_secrets"))
    tier_report = conf.get("tier_report") or {}

    if env_stem and _config_dir.is_dir():
        from dotenv import dotenv_values
        # An explicit overlay entry stays authoritative over the env file.
        for key, value in dotenv_values(_config_dir / f"env.{env_stem}").items():
            overlay.setdefault(key, value)

    from migrator_utils.migrations.shared import get_config

    with _env_overlay(overlay):
        resolved = get_config()
        env_seen = {
            key: os.getenv(key) if reveal or not _is_secret(key)
            else _fingerprint(os.getenv(key))
            for key in report_env
        }

    payload = {
        "run_id": ctx.get("run_id"),
        "context_module": ctx_module,
        "conf": {key: value for key, value in conf.items() if key != "env_overlay"},
        "overlay_keys": sorted(overlay),
        "env_seen": env_seen,
        "diagnostics": _diagnostics(),
        "config": _redact(resolved, reveal),
        "secrets_revealed": reveal,
        "tiers": _tier_snapshot(
            tier_report,
            ctx.get("run_id"),
            str(conf.get("triggered_by", "")).strip().lower() == "portal",
            reveal,
        ),
    }

    text = json.dumps(payload, default=str, sort_keys=True)
    if reveal:
        # XCom alone carries the plaintext. Task logs are shipped to the log
        # backend and kept after the run is deleted, so they must not get it.
        print(f"{PAYLOAD_BEGIN} withheld from the log: reveal_secrets is on")
    else:
        print(PAYLOAD_BEGIN)
        print(text)
        print(PAYLOAD_END)
    return text


with DAG(
    dag_id="nx1_var_probe",
    description="TEST DAG - reports get_config() resolution. Safe to delete.",
    schedule=None,
    start_date=datetime(2024, 1, 1),
    catchup=False,
    max_active_runs=8,
    tags=["nx1", "test", "variable-precedence"],
    default_args={"owner": "data-migration", "retries": 0},
) as dag:
    probe()
'''

print(f"Probe DAG source: {len(PROBE_DAG_SOURCE)} bytes, {len(PROBE_DAG_SOURCE.splitlines())} lines")

In [ ]:
# ── Upload the probe DAG and wait for Airflow to register it ─────────────────────
if _missing:
    raise RuntimeError(f"Set these first: {', '.join(_missing)}")

written = s3_put_text(PROBE_S3_URI, PROBE_DAG_SOURCE)
print(f"Uploaded {written} bytes -> {PROBE_S3_URI}")

DEADLINE = time.time() + 600
while time.time() < DEADLINE:
    dag = af("GET", f"/dags/{PROBE_DAG_ID}", allow=(404,))
    if dag is not None:
        print(f"Registered: {PROBE_DAG_ID} (paused={dag.get('is_paused')})")
        break
    print("  waiting for the DAG bundle to sync...")
    time.sleep(15)
else:
    raise RuntimeError(
        f"{PROBE_DAG_ID} did not appear within 10 min. Check the DAG-bundle sync "
        f"interval and that {PROBE_S3_KEY} is inside the folder Airflow reads."
    )

if dag.get("is_paused"):
    af("PATCH", f"/dags/{PROBE_DAG_ID}", json={"is_paused": False})
    print("Unpaused.")

# An import error here means the deployed shared.py or its deps are broken — every
# real migration DAG would be failing the same way.
errors = af("GET", "/importErrors", allow=(404,)) or {}
for entry in errors.get("import_errors", []):
    if PROBE_DAG_ID in (entry.get("filename") or ""):
        raise RuntimeError(f"Probe DAG import error:\n{entry.get('stack_trace')}")
print("No import errors for the probe DAG.")

---
## Step 2 — Safety preflight

The matrix needs plain Variables such as `auth_method` and `cluster_type` set to specific
values (including empty), and it needs the `nx1_s3_*` globals that Infrastructure Config owns.
Those are tenant-wide: a migration DAG running concurrently would resolve the probe's values.

This cell refuses to continue while any migration DAG run is active, and shows the current
value of every key about to be touched so you can see what is being borrowed.

In [ ]:
# ── Refuse to run alongside a live migration ─────────────────────────────────────
active = []
listing = af("GET", "/dags", params={"limit": 1000}) or {}
returned, total = len(listing.get("dags", [])), listing.get("total_entries", 0)
if total > returned:
    raise RuntimeError(
        f"Only {returned} of {total} DAGs listed — the guard below could miss a running "
        f"migration. Page through /dags before continuing."
    )
for entry in listing.get("dags", []):
    dag_id = entry.get("dag_id", "")
    if not dag_id.startswith(GUARDED_DAG_PREFIXES):
        continue
    runs = af("GET", f"/dags/{dag_id}/dagRuns",
              params={"limit": 5, "state": ["running", "queued"]}, allow=(400, 404))
    for run in (runs or {}).get("dag_runs", []):
        active.append(f"{dag_id} / {run.get('dag_run_id')} ({run.get('state')})")

if active:
    raise RuntimeError(
        "Migration DAG runs are in flight — this notebook would change the Variables "
        "they resolve. Wait for them to finish:\n  " + "\n  ".join(active)
    )
print("No migration DAG runs active.")

---
## Inspect the stored credentials (read-only, standalone)

Prints what is actually stored for the S3 keys, at every tier, including the real values.
Writes nothing and touches no Variable — run it on its own without the rest of the notebook.

This is the check for "did the portal corrupt my credentials": if a value reads back as `***`
then something wrote Airflow's mask over it. Anything else is a real value.

Requires the probe DAG from Step 1 (it carries the tier snapshot). The plaintext lands in that
run's XCom, which the last line deletes; it is deliberately kept out of the task log.

In [ ]:
CREDENTIAL_KEYS = {"s3_access_key": "S3_ACCESS_KEY",
                   "s3_secret_key": "S3_SECRET_KEY",
                   "s3_endpoint": "S3_ENDPOINT"}

inspect_id = f"{PROBE_DAG_ID}_inspect_{datetime.now(timezone.utc).strftime('%H%M%S')}"
trigger(inspect_id, {"triggered_by": "portal", "reveal_secrets": True,
                     "tier_report": CREDENTIAL_KEYS})
wait_for([inspect_id])
inspected = fetch_payload(inspect_id)

if not inspected.get("tiers"):
    print("No tier snapshot — the deployed probe DAG predates it. Re-run Step 1, wait for "
          "the bundle to resync, then re-run this cell.")
else:
    TIER_ORDER = [("run_scoped", "1 run-scoped"), ("nx1", "2 nx1_ global"),
                  ("plain", "3 plain Variable"), ("env", "4 env file")]
    for base_key, entry in inspected["tiers"].items():
        resolved = inspected["config"].get(base_key)
        print(f"\n{base_key}")
        print(f"  resolved to  {resolved!r}")
        for tier, label in TIER_ORDER:
            candidate = entry.get(tier) or {}
            if candidate.get("error"):
                print(f"  tier {label:<17} {candidate['source']}: "
                      f"LOOKUP FAILED {candidate['error']}")
            elif candidate.get("present"):
                flag = "  <-- MASK, this value is corrupted" if candidate["value"] == MASK else ""
                gate = "" if candidate.get("eligible", True) else "  (portal runs only)"
                print(f"  tier {label:<17} {candidate['source']}: "
                      f"{candidate['value']!r}{gate}{flag}")
            else:
                print(f"  tier {label:<17} {candidate.get('source') or '-'}: (not set)")

purged, failed_purge = purge_probe_xcoms()
print(f"\nDeleted {len(purged)} probe run(s) and their XComs"
      + (f" — failed: {failed_purge}" if failed_purge else ""))

---
## Step 3 — The scenario matrix

Each scenario owns one config key, so a single DAG run evaluates all of them. Columns
`t1`–`t4` are what gets written at each tier:

- `set` → a `probe-*` value
- `''` → the Variable exists and is **empty** (the case that used to shadow everything below)
- `-` → the Variable is **absent** (deleted first, in case the tenant already had one)

`expect(portal)` and `expect(manual)` are written out by hand rather than computed from the
tiers — a harness that re-derives the rule under test would agree with a broken `_var`.

`DEFAULT` means the hardcoded fallback in `get_config`, listed in `KEY_ENV_DEFAULT` below.
That table and `CONFIG_KEY` mirror `get_config` and have to be updated alongside it.

### Which scenarios test the change, and which guard against regressions

Replaying this matrix against `main`'s `shared.py` fails 28 of the 104 scenario/run
combinations, in these ten scenarios — they are what the branch changes:

`T2-WINS` `T2-ONLY` `T2-ISOLATION` `T2-OVER-3-AND-4` `S3-ENDPOINT` `S3-ACCESS-KEY`
`T3-EMPTY` `T3-EMPTY-TO-DEFAULT` `T5-ALL-EMPTY` `BOOL-T3-EMPTY`

The other sixteen pass on `main` too and must keep passing. The `T1-*` group in particular:
`main` already resolved the run-scoped tier on presence, so tier 1 behaviour is unchanged by
this branch and those rows are guards, not new-behaviour checks.

In [ ]:
# ── Key tables (mirror get_config in migrator_utils/migrations/shared.py) ────────
from collections import namedtuple

# base Variable key -> (env var, hardcoded default)
KEY_ENV_DEFAULT = {
    "cluster_ssh_conn_id":              ("CLUSTER_SSH_CONN_ID", "cluster_edge_ssh"),
    "cluster_edge_temp_path":           ("CLUSTER_EDGE_TEMP_PATH", "/tmp/migration"),
    "cluster_edge_discovery_temp_path": ("CLUSTER_EDGE_DISCOVERY_TEMP_PATH", "/tmp"),
    "cluster_hive_scratch_dir":         ("CLUSTER_HIVE_SCRATCH_DIR", "/tmp/hive"),
    "cluster_distcp_log_root":          ("CLUSTER_DISTCP_LOG_ROOT", "/tmp"),
    "migration_default_s3_bucket":      ("MIGRATION_DEFAULT_S3_BUCKET", "s3a://data-lake"),
    "s3_endpoint":                      ("S3_ENDPOINT", ""),
    "s3_access_key":                    ("S3_ACCESS_KEY", ""),
    "s3_secret_key":                    ("S3_SECRET_KEY", ""),
    "migration_distcp_bandwidth":       ("MIGRATION_DISTCP_BANDWIDTH", "100"),
    "migration_distcp_preserve_delete": ("MIGRATION_DISTCP_PRESERVE_DELETE", "true"),
    "migration_spark_conn_id":          ("MIGRATION_SPARK_CONN_ID", "spark_default"),
    "migration_tracking_database":      ("MIGRATION_TRACKING_DATABASE", "migration_tracking"),
    "migration_tracking_location":      ("MIGRATION_TRACKING_LOCATION", "s3a://data-lake/migration_tracking"),
    "migration_report_location":        ("MIGRATION_REPORT_LOCATION", "s3a://data-lake/migration_reports"),
    "cluster_type":                     ("CLUSTER_TYPE", "MapR"),
    "auth_method":                      ("AUTH_METHOD", "mapr"),
    "mapr_ticketfile_location":         ("MAPR_TICKETFILE_LOCATION", "/tmp/maprticket_${USER}"),
    "hdfs_nameservice":                 ("HDFS_NAMESERVICE", ""),
    "s3_listing_tool":                  ("S3_LISTING_TOOL", "hadoop"),
    "s3_source_endpoint":               ("S3_SOURCE_ENDPOINT", ""),
    "s3_dest_endpoint":                 ("S3_DEST_ENDPOINT", ""),
    "migration_smtp_conn_id":           ("MIGRATION_SMTP_CONN_ID", "smtp_default"),
    "migration_email_recipients":       ("MIGRATION_EMAIL_RECIPIENTS", ""),
    "migration_include_db_in_path":     ("MIGRATION_INCLUDE_DB_IN_PATH", "true"),
    "migration_recreate_tables":        ("MIGRATION_RECREATE_TABLES", "false"),
    "migration_dag_owner":              ("MIGRATION_DAG_OWNER", ""),
}

# base Variable key -> key in the dict get_config returns
CONFIG_KEY = {
    "cluster_ssh_conn_id":              "ssh_conn_id",
    "cluster_edge_temp_path":           "edge_temp_path",
    "cluster_edge_discovery_temp_path": "edge_discovery_temp_path",
    "cluster_hive_scratch_dir":         "hive_scratch_dir",
    "cluster_distcp_log_root":          "distcp_log_root",
    "migration_default_s3_bucket":      "default_s3_bucket",
    "s3_endpoint":                      "s3_endpoint",
    "s3_access_key":                    "s3_access_key",
    "s3_secret_key":                    "s3_secret_key",
    "migration_distcp_bandwidth":       "distcp_bandwidth",
    "migration_distcp_preserve_delete": "distcp_preserve_delete",
    "migration_spark_conn_id":          "spark_conn_id",
    "migration_tracking_database":      "tracking_database",
    "migration_tracking_location":      "tracking_location",
    "migration_report_location":        "report_output_location",
    "cluster_type":                     "cluster_type",
    "auth_method":                      "auth_method",
    "mapr_ticketfile_location":         "mapr_ticketfile_location",
    "hdfs_nameservice":                 "hdfs_nameservice",
    "s3_listing_tool":                  "s3_listing_tool",
    "s3_source_endpoint":               "s3_source_endpoint",
    "s3_dest_endpoint":                 "s3_dest_endpoint",
    "migration_smtp_conn_id":           "smtp_conn_id",
    "migration_email_recipients":       "email_recipients",
    "migration_include_db_in_path":     "include_db_in_path",
    "migration_recreate_tables":        "recreate_tables",
    "migration_dag_owner":              "owner",
}

# get_config coerces these to bool with the same truthy-token list
BOOL_KEYS = {"migration_distcp_preserve_delete", "migration_include_db_in_path",
             "migration_recreate_tables"}
TRUTHY = ("1", "true", "yes", "y", "on")


def as_bool(value):
    return str(value).strip().lower() in TRUTHY


DEFAULT = "<<hardcoded default>>"

Scenario = namedtuple("Scenario", "sid title key t1 t2 t3 t4 portal manual note")


def S(sid, title, key, t1, t2, t3, t4, portal, manual, note=""):
    return Scenario(sid, title, key, t1, t2, t3, t4, portal, manual, note)


SCENARIOS = [
    # ---- tier 1: run-scoped, written only by the portal, matched on presence ----
    S("T1-ALL", "tier 1 outranks every tier below it",
      "cluster_ssh_conn_id", "probe-t1", "probe-nx1", "probe-plain", "probe-env",
      "probe-t1", "probe-t1"),
    S("T1-EMPTY", "an empty tier 1 is authoritative — blank means 'send no report'",
      "migration_email_recipients", "", "probe-nx1", "probe-plain", "probe-env",
      "", ""),
    S("T1-EMPTY-BARE", "an empty tier 1 wins even with a value below it",
      "cluster_edge_discovery_temp_path", "", None, None, "probe-env",
      "", ""),
    S("T1-WHITESPACE", "tier 1 passes whitespace through unchanged",
      "cluster_edge_temp_path", " ", None, "probe-plain", "probe-env",
      " ", " "),
    S("T1-NOT-GATED", "tier 1 is not gated on the portal marker",
      "hdfs_nameservice", "probe-t1", None, "probe-plain", None,
      "probe-t1", "probe-t1"),
    S("OWNER-T1", "tier 1 feeds the owner fallback chain",
      "migration_dag_owner", "probe-owner-t1", None, None, None,
      "probe-owner-t1", "probe-owner-t1"),

    # ---- tier 2: nx1_ namespace, portal-triggered runs only ----
    S("T2-WINS", "tier 2 outranks plain Variables and env — portal only",
      "migration_tracking_database", None, "probe-nx1", "probe-plain", "probe-env",
      "probe-nx1", "probe-plain"),
    S("T2-EMPTY", "an empty tier 2 falls through to the plain Variable",
      "migration_tracking_location", None, "", "probe-plain", "probe-env",
      "probe-plain", "probe-plain"),
    S("T2-ONLY", "tier 2 alone: the portal sees it, a manual run gets the default",
      "migration_report_location", None, "probe-nx1", None, None,
      "probe-nx1", DEFAULT),
    S("T2-ISOLATION", "the nx1_ namespace stays invisible to a hand-launched run",
      "s3_dest_endpoint", None, "probe-nx1", None, None,
      "probe-nx1", DEFAULT),
    S("T2-OVER-3-AND-4", "tier 2 beats tier 3 and tier 4 together",
      "migration_smtp_conn_id", None, "probe-nx1", "probe-plain", "probe-env",
      "probe-nx1", "probe-plain"),

    # ---- tier 3: plain Variables ----
    S("T3-WINS", "tier 3 wins when tiers 1 and 2 are absent",
      "cluster_type", None, None, "probe-plain", "probe-env",
      "probe-plain", "probe-plain"),
    S("T3-EMPTY", "an empty plain Variable no longer masks env (the shadowing bug)",
      "auth_method", None, None, "", "probe-env",
      "probe-env", "probe-env"),
    S("T3-EMPTY-TO-DEFAULT", "an empty plain Variable with no env falls to the default",
      "cluster_distcp_log_root", None, None, "", None,
      DEFAULT, DEFAULT),
    S("T3-NUMERIC", "a numeric value passes through as a string",
      "migration_distcp_bandwidth", None, None, "222", "333",
      "222", "222"),

    # ---- tiers 4 and 5 ----
    S("T4-ENV", "tier 4 env value when nothing above it is set",
      "mapr_ticketfile_location", None, None, None, "probe-env",
      "probe-env", "probe-env"),
    S("T5-DEFAULT", "the hardcoded default when no tier is set",
      "cluster_hive_scratch_dir", None, None, None, None,
      DEFAULT, DEFAULT),
    S("T5-ALL-EMPTY", "empty at tiers 2 and 3 both fall through to the default",
      "migration_spark_conn_id", None, "", "", None,
      DEFAULT, DEFAULT),

    # ---- the three S3 globals this change introduces ----
    S("S3-ENDPOINT", "s3_endpoint inherits the tenant global on a portal run",
      "s3_endpoint", None, "https://probe-nx1.example.invalid", None,
      "https://probe-env.example.invalid",
      "https://probe-nx1.example.invalid", "https://probe-env.example.invalid"),
    S("S3-ACCESS-KEY", "s3_access_key inherits the tenant global on a portal run",
      "s3_access_key", None, "probe-nx1-access", None, "probe-env-access",
      "probe-nx1-access", "probe-env-access"),
    S("S3-SECRET-KEY", "a run override outranks the tenant global secret",
      "s3_secret_key", "probe-t1-secret", "probe-nx1-secret", None, "probe-env-secret",
      "probe-t1-secret", "probe-t1-secret"),
    S("S3-RETIRED-KEYS", "the retired s3_source_* keys still resolve for the DAGs reading them",
      "s3_source_endpoint", "probe-t1", "probe-nx1", None, None,
      "probe-t1", "probe-t1",
      note="No portal field writes these any more; the DAG still reads them."),

    # ---- bool coercion ----
    S("BOOL-T3-EMPTY", "an empty plain bool falls to the 'true' default",
      "migration_distcp_preserve_delete", None, None, "", None,
      True, True,
      note="Documented behaviour change: this used to resolve False."),
    S("BOOL-ENV-FALSE", "an empty plain bool falls through to a false env value",
      "migration_include_db_in_path", None, None, "", "false",
      False, False),
    S("BOOL-T1-EMPTY", "an empty tier 1 bool resolves False, not the nx1_ 'true'",
      "migration_recreate_tables", "", "true", None, None,
      False, False),

    # ---- documented exception ----
    S("LISTING-TOOL", "s3_listing_tool bypasses _var — no tier 1, no nx1_",
      "s3_listing_tool", "probe-t1", "probe-nx1", None, "probe-env",
      "probe-env", "probe-env",
      note="Reads Variable.get(key, default_var=os.getenv(...)) directly. "
           "Pre-existing; this branch does not change it."),
]

assert len(SCENARIOS) == len({s.key for s in SCENARIOS}), \
    "each scenario needs its own carrier key, or they overwrite each other"
assert not set(KEY_ENV_DEFAULT) ^ set(CONFIG_KEY), "KEY_ENV_DEFAULT and CONFIG_KEY disagree"

# One env overlay for the whole matrix: None means "unset inside the task", so the
# scenarios are independent of whatever this tenant happens to have in env.shared.
ENV_OVERLAY = {KEY_ENV_DEFAULT[s.key][0]: s.t4 for s in SCENARIOS}


def expected(scenario, portal):
    want = scenario.portal if portal else scenario.manual
    if want == DEFAULT:
        want = KEY_ENV_DEFAULT[scenario.key][1]
    if scenario.key in BOOL_KEYS and not isinstance(want, bool):
        return as_bool(want)
    return want


def show(value):
    if value is None:
        return "-"
    if value == DEFAULT:
        return "default"
    if isinstance(value, str) and not value:
        return "''"
    if isinstance(value, str) and not value.strip():
        return repr(value)
    return str(value)


print(f"{len(SCENARIOS)} scenarios\n")
head = f"{'scenario':<20} {'carrier key':<34} {'t1':<10} {'t2':<10} {'t3':<10} {'t4':<10} " \
       f"{'portal':<12} {'manual':<12}"
print(head)
print("-" * len(head))
for s in SCENARIOS:
    print(f"{s.sid:<20} {s.key:<34} {show(s.t1):<10} {show(s.t2):<10} {show(s.t3):<10} "
          f"{show(s.t4):<10} {show(s.portal):<12} {show(s.manual):<12}")
print("\nEnv overlay applied inside the task:")
for env_var, value in sorted(ENV_OVERLAY.items()):
    print(f"  {env_var:<36} {'(unset)' if value is None else value}")

---
## Step 4 — Baseline run, before anything is touched

Resolves the config with **no test Variables and no env overlay** — what a real portal-triggered
run gets on this tenant right now. Step 9 repeats it after cleanup and diffs the two, which is
how a leaked Variable gets caught.

The baseline uses the portal marker deliberately: a portal run can see tiers 1–5, so it detects
leftovers in the `nx1_` namespace as well as in plain Variables. A manual run could not.

In [ ]:
RUN_STAMP = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")


def run_id_for(label):
    return f"{PROBE_DAG_ID}_{RUN_STAMP}_{label}"


REPORT_ENV = sorted({KEY_ENV_DEFAULT[s.key][0] for s in SCENARIOS})
# Ask the probe what sits at every tier for each key, so the printout below can
# name the Airflow Variable a value actually came from — nx1_s3_access_key versus
# the plain s3_access_key, for instance.
TIER_REPORT = {key: env_var for key, (env_var, _) in KEY_ENV_DEFAULT.items()}

BASELINE_CONF = {"triggered_by": "portal", "report_env": REPORT_ENV,
                 "reveal_secrets": REVEAL_SECRETS, "tier_report": TIER_REPORT}
if BASELINE_ENV_STEM:
    BASELINE_CONF["env_stem"] = BASELINE_ENV_STEM

baseline_pre_id = run_id_for("baseline_pre")
trigger(baseline_pre_id, BASELINE_CONF)
print(f"Triggered {baseline_pre_id}")
wait_for([baseline_pre_id])

BASELINE_PRE = fetch_payload(baseline_pre_id)
diag = BASELINE_PRE["diagnostics"]

print("\n── Deployed code ─────────────────────────────────────────────")
print(f"  shared.py            : {diag['shared_py_path']}")
print(f"  md5 / bytes          : {diag['shared_py_md5']} / {diag['shared_py_bytes']}")
print(f"  Airflow              : {diag['airflow_version']}")
print(f"  PORTAL_TRIGGER       : {diag['portal_trigger_const']!r}")
print(f"  nx1_ tier on disk    : {diag['has_nx1_tier']}")
print(f"  nx1_ tier in RAM     : {diag['loaded_code_has_nx1_tier']}")
print(f"  tier 1 on presence   : {diag['tier1_on_presence']}")
print(f"  swallows lookup errs : {diag['swallows_lookup_errors']}")
print(f"  run_id seen in task  : {BASELINE_PRE['run_id']!r}")
print(f"  context module       : {BASELINE_PRE['context_module']!r}")
print(f"  env.shared present   : {diag['env_shared_dir_present']}")

TIER_LABELS = {
    "run_scoped": "tier 1 run-scoped",
    "nx1": "tier 2 nx1_",
    "plain": "tier 3 plain",
    "env": "tier 4 env",
}


def render_value(value):
    if isinstance(value, dict) and value.get("redacted"):
        return f"<redacted sha {value['sha256_8']} len {value['len']}>"
    return repr(value)


def sources_matching(base_key, resolved, tiers):
    """Which tiers hold a value equal to what got resolved.

    More than one hit means the source is ambiguous — the tiers agree, so the
    resolved value cannot tell you which one won. That is worth seeing rather
    than papering over.
    """
    entry = tiers.get(base_key) or {}
    hits = []
    for tier in ("run_scoped", "nx1", "plain", "env"):
        candidate = entry.get(tier) or {}
        if candidate.get("error"):
            hits.append(f"{TIER_LABELS[tier]} {candidate['source']} LOOKUP FAILED")
            continue
        if not candidate.get("present"):
            continue
        value = candidate["value"]
        same = (
            as_bool(value) == resolved
            if base_key in BOOL_KEYS and isinstance(resolved, bool)
            else value == resolved
        )
        if not same:
            continue
        note = ""
        if tier == "nx1" and not candidate.get("eligible", True):
            note = " (set, but this run is not portal-triggered)"
        hits.append(f"{TIER_LABELS[tier]} {candidate['source']}{note}")
    return hits


BASE_KEY_FOR_CONFIG_KEY = {config: base for base, config in CONFIG_KEY.items()}

print("\n── Resolved config (as a real portal run sees it now) ────────")
print("   'source' names the Airflow Variable or env var holding that exact value.")
for key, value in sorted(BASELINE_PRE["config"].items()):
    base_key = BASE_KEY_FOR_CONFIG_KEY.get(key)
    hits = (
        sources_matching(base_key, value, BASELINE_PRE.get("tiers", {}))
        if base_key
        else []
    )
    if hits:
        source = "; ".join(hits)
    elif base_key:
        source = "tier 5 hardcoded default (no tier holds this value)"
    else:
        source = "not resolved through _var"
    print(f"  {key:<28} {render_value(value):<42} {source}")

In [ ]:
# ── The deployed code has to be the version under test ──────────────────────────
# Without these the matrix would report confusing failures instead of "stale deploy".
problems = []
if diag["portal_trigger_const"] != "portal":
    problems.append(f"PORTAL_TRIGGER is {diag['portal_trigger_const']!r}, expected 'portal'")
if not diag["has_nx1_tier"]:
    problems.append("deployed _var has no nx1_ tier — old shared.py is live")
if not diag["loaded_code_has_nx1_tier"]:
    problems.append(
        "the shared.py on disk has the nx1_ tier but the code get_config is running "
        "does not — a worker imported it before the deploy and sys.modules is stale. "
        "Restart the scheduler/workers, or wait for the pods to cycle."
    )
if not diag["tier1_on_presence"]:
    problems.append("deployed _var still matches tier 1 on truthiness, not presence")
if diag["swallows_lookup_errors"]:
    problems.append("deployed _var still wraps Variable.get in except Exception")
if not BASELINE_PRE["run_id"]:
    problems.append(
        "get_config saw no run_id — get_current_context failed, so tiers 1 and 2 "
        "cannot resolve at all on this Airflow version"
    )
if not diag["env_shared_dir_present"]:
    problems.append(
        "migrator_utils/migration_configs is missing from the DAG folder — tier 4 "
        "is empty for every DAG, not just the probe"
    )

if problems:
    raise RuntimeError("Deployed code is not the version under test:\n  - "
                       + "\n  - ".join(problems))
print("Preflight OK — deployed shared.py carries the tiers this notebook tests.")

---
## Step 5 — Write the matrix Variables

`VariableSandbox` records the prior value of every global key before its first write, so
cleanup can put the tenant back exactly — including deleting keys that did not exist. Scenarios
whose tier is `-` need the Variable **absent**, so an existing one is deleted too.

Run-scoped keys are per run id and unique to this notebook, so they only need deleting, not
restoring.

The backup file is written on every claim (mode 600) and holds real secret values; cleanup
deletes it.

In [ ]:
class VariableSandbox:
    """Borrow tenant-wide Airflow Variables and give them back unchanged.

    Records each key's prior value — or None for "did not exist" — on first touch,
    so restore() can delete keys it created rather than leaving them behind.
    """

    def __init__(self, path):
        self.path = path
        self.saved = {}
        # A previous run that never reached cleanup has already replaced these
        # Variables with scenario values. Re-reading them now would record the
        # probe value as the "prior" one and make the real value unrecoverable, so
        # an existing backup always wins.
        if os.path.exists(path):
            with open(path) as handle:
                self.saved = json.load(handle)
            print(f"  reusing {len(self.saved)} prior values from {path} — a previous "
                  f"run did not clean up. Run the cleanup cell before starting over.")

    def _claim(self, key):
        if key not in self.saved:
            value = var_get(key)
            if value == MASK:
                # Airflow masks credential-shaped Variables in its API. Saving the
                # mask and writing it back on restore would replace a real secret
                # with the literal '***'. Refuse rather than corrupt it.
                raise RuntimeError(
                    f"{key} reads back as {MASK!r} — Airflow is masking it, so its "
                    f"real value cannot be recovered through the API. Add it to "
                    f"KNOWN_VALUES so cleanup can restore it, or delete the Variable "
                    f"by hand if it is not needed."
                )
            self.saved[key] = value
            self._persist()

    def set(self, key, value):
        self._claim(key)
        var_set(key, value)

    def delete(self, key):
        self._claim(key)
        var_delete(key)

    def restore(self):
        restored, removed, failed = [], [], []
        for key, value in sorted(self.saved.items()):
            try:
                if value is None:
                    var_delete(key)
                    removed.append(key)
                else:
                    var_set(key, value)
                    restored.append(key)
            except Exception as exc:
                failed.append((key, str(exc)))
        return restored, removed, failed

    def _persist(self):
        with open(self.path, "w") as handle:
            json.dump(self.saved, handle, indent=2, sort_keys=True)
        os.chmod(self.path, 0o600)


# True values for keys Airflow masks, so the sandbox can restore them. Anything set
# here by hand wins; recover_masked_values() fills in the rest. Values never leave
# this notebook; only fingerprints are printed.
KNOWN_VALUES: dict[str, str] = {
    # "s3_access_key": "...",
}


def _parse_env_file(text):
    """Minimal KEY=VALUE reader — python-dotenv is not guaranteed on a JH image."""
    out = {}
    for raw in text.splitlines():
        line = raw.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, _, value = line.partition("=")
        out[key.strip()] = value.strip().strip('"').strip("'")
    return out


def recover_masked_values():
    """Recover masked credentials from the env.shared deployed beside the DAGs.

    Airflow will not return these through its API, and typing them in would leave
    real credentials in the committed .ipynb. env.shared holds the same ones — but
    only where nothing has drifted, so each candidate is accepted only if the
    fingerprint baseline_pre recorded at that tier matches. A key whose stored
    value differs is left out, and the mask scan below then stops the run rather
    than restoring the wrong credential.
    """
    uri = (f"s3a://{DAGS_S3_BUCKET}/{DAGS_S3_PREFIX}"
           f"migrator_utils/migration_configs/env.shared")
    try:
        env = _parse_env_file(s3_read_text(uri))
    except Exception as exc:
        print(f"  could not read {uri}: {exc}")
        return {}

    tiers = BASELINE_PRE.get("tiers", {})
    if not tiers:
        print("  baseline_pre carries no tier snapshot — the probe DAG deployed on the "
              "tenant predates it. Re-run Step 1 to upload the current one, wait for "
              "the DAG bundle to resync, then re-run Step 4 before this cell.")
        return {}
    revealed = BASELINE_PRE.get("secrets_revealed", False)
    recovered, rejected = {}, []
    for base, env_name in (("s3_access_key", "S3_ACCESS_KEY"),
                           ("s3_secret_key", "S3_SECRET_KEY")):
        value = env.get(env_name)
        if not value:
            continue
        expected = value if revealed else fingerprint(value)
        for key, tier in ((base, "plain"), (f"nx1_{base}", "nx1")):
            recorded = ((tiers.get(base) or {}).get(tier) or {})
            if not recorded.get("present"):
                continue           # absent, so restore is a delete — nothing needed
            if recorded.get("value") == expected:
                recovered[key] = value
            else:
                rejected.append(key)

    for key in recovered:
        print(f"  recovered {key} from env.shared (fingerprint matches what "
              f"baseline_pre saw at that tier)")
    for key in rejected:
        print(f"  REJECTED {key}: stored value differs from env.shared — supply it "
              f"in KNOWN_VALUES by hand")
    return recovered


print("Recovering values Airflow will not hand back:")
for key, value in recover_masked_values().items():
    KNOWN_VALUES.setdefault(key, value)
print(f"  KNOWN_VALUES now covers: {sorted(KNOWN_VALUES) or 'nothing'}\n")

SANDBOX = VariableSandbox(VARIABLE_BACKUP_PATH)

MATRIX_RUNS = {
    "portal":           ({"triggered_by": "portal"}, True),
    "portal_mixedcase": ({"triggered_by": "  PoRtAl  "}, True),
    "manual":           ({}, False),
    "other_marker":     ({"triggered_by": "manual"}, False),
}
MATRIX_RUN_IDS = {label: run_id_for(label) for label in MATRIX_RUNS}
SCOPED_KEYS_WRITTEN = []

# Fail before the first write: find every key whose real value the API will not
# give back, so the run cannot get half-way and then corrupt it on restore.
to_claim = [f"{prefix}{s.key}" for s in SCENARIOS for prefix in ("nx1_", "")]
masked = [key for key in to_claim if key not in KNOWN_VALUES and var_get(key) == MASK]
if masked:
    raise RuntimeError(
        "Airflow masks these Variables in its API, so their real values cannot be "
        "read and restored:\n  " + "\n  ".join(masked)
        + "\n\nNothing has been written yet. Add each one's true value to "
          "KNOWN_VALUES above (from the tenant's env.shared), or delete the Variable "
          "in Airflow if it is not needed, then re-run this cell."
    )
print(f"Mask scan clear — all {len(to_claim)} keys are readable or listed in KNOWN_VALUES.\n")

for key, value in KNOWN_VALUES.items():
    SANDBOX.saved.setdefault(key, value)

for scenario in SCENARIOS:
    for prefix, value in (("nx1_", scenario.t2), ("", scenario.t3)):
        key = f"{prefix}{scenario.key}"
        if value is None:
            SANDBOX.delete(key)
        else:
            SANDBOX.set(key, value)
    if scenario.t1 is not None:
        for run_id in MATRIX_RUN_IDS.values():
            scoped = f"{scenario.key}__{run_id}"
            var_set(scoped, scenario.t1)
            SCOPED_KEYS_WRITTEN.append(scoped)

print(f"Borrowed {len(SANDBOX.saved)} tenant Variables "
      f"(backup: {VARIABLE_BACKUP_PATH})")
pre_existing = {k: v for k, v in SANDBOX.saved.items() if v is not None}
print(f"  {len(pre_existing)} already existed and will be restored:")
for key, value in sorted(pre_existing.items()):
    rendered = (
        repr(value) if REVEAL_SECRETS or not is_secret(key)
        else f"<redacted sha {fingerprint(value)['sha256_8']}>"
    )
    print(f"    {key:<40} {rendered}")
print(f"  {len(SANDBOX.saved) - len(pre_existing)} did not exist and will be deleted again")
print(f"\nWrote {len(SCOPED_KEYS_WRITTEN)} run-scoped Variables across "
      f"{len(MATRIX_RUN_IDS)} runs")

---
## Step 6 — Trigger the matrix runs

All four fire at once; the probe DAG allows 8 concurrent runs. `portal_mixedcase` and
`other_marker` prove the marker is compared after `.strip().lower()` and that an arbitrary
marker does **not** unlock the `nx1_` tier.

In [ ]:
for label, (conf, _) in MATRIX_RUNS.items():
    body = dict(conf)
    body["env_overlay"] = ENV_OVERLAY
    body["report_env"] = REPORT_ENV
    body["reveal_secrets"] = REVEAL_SECRETS
    if BASELINE_ENV_STEM:
        body["env_stem"] = BASELINE_ENV_STEM
    trigger(MATRIX_RUN_IDS[label], body)
    print(f"Triggered {MATRIX_RUN_IDS[label]}  conf.triggered_by={conf.get('triggered_by')!r}")

print()
states = wait_for(list(MATRIX_RUN_IDS.values()))

PAYLOADS = {}
for label, run_id in MATRIX_RUN_IDS.items():
    if states.get(run_id) != "success":
        raise RuntimeError(
            f"{run_id} ended {states.get(run_id)} — open its task log in Airflow. "
            f"A failure here is itself a result: get_config raised instead of resolving."
        )
    PAYLOADS[label] = fetch_payload(run_id)
    seen = PAYLOADS[label]["run_id"]
    if seen != run_id:
        raise RuntimeError(f"{label}: task saw run_id {seen!r}, expected {run_id!r} — "
                           f"tier 1 keys would not match")
print(f"\nCollected {len(PAYLOADS)} payloads.")

---
## Step 7 — Evaluate

One row per scenario per run. `expected` comes from the hand-written matrix; `actual` is what
the deployed `get_config()` returned. Credential-shaped keys are compared as fingerprints.

In [ ]:
def render(value):
    if isinstance(value, dict) and value.get("redacted"):
        return f"sha {value['sha256_8']}/{value['len']}"
    if isinstance(value, str) and not value:
        return "''"
    if isinstance(value, str) and not value.strip():
        return repr(value)
    return str(value)


results = []
for label, (_, portal) in MATRIX_RUNS.items():
    config = PAYLOADS[label]["config"]
    for scenario in SCENARIOS:
        config_key = CONFIG_KEY[scenario.key]
        actual = config.get(config_key)
        want = expected(scenario, portal)
        # The probe fingerprints credential-shaped values unless REVEAL_SECRETS
        # is on, so the expected side has to be shaped the same way.
        revealed = PAYLOADS[label].get("secrets_revealed", False)
        want_cmp = (
            want if revealed or not is_secret(config_key) else fingerprint(want)
        )
        results.append({
            "run": label, "sid": scenario.sid, "key": scenario.key,
            "expected": want_cmp, "actual": actual, "ok": actual == want_cmp,
            "title": scenario.title, "note": scenario.note,
        })

header = f"{'':<4} {'scenario':<20} {'carrier key':<34} {'expected':<38} {'actual':<38}"
for label in MATRIX_RUNS:
    rows = [row for row in results if row["run"] == label]
    marker = MATRIX_RUNS[label][0].get("triggered_by")
    tier = "portal (tiers 1-5)" if MATRIX_RUNS[label][1] else "manual (nx1_ invisible)"
    print(f"\n=== {label} — conf.triggered_by={marker!r} -> {tier} ===")
    print(header)
    print("-" * len(header))
    for row in rows:
        flag = "PASS" if row["ok"] else "FAIL"
        print(f"{flag:<4} {row['sid']:<20} {row['key']:<34} "
              f"{render(row['expected']):<38} {render(row['actual']):<38}")

failed = [row for row in results if not row["ok"]]
print(f"\n{'=' * 96}")
print(f"{len(results) - len(failed)}/{len(results)} passed")
if failed:
    print(f"\n{len(failed)} FAILED:")
    for row in failed:
        print(f"  [{row['run']}] {row['sid']} ({row['key']}): {row['title']}")
        print(f"      expected {render(row['expected'])!r}, got {render(row['actual'])!r}")
        env_var = KEY_ENV_DEFAULT[row["key"]][0]
        print(f"      env {env_var} in task = "
              f"{render(PAYLOADS[row['run']]['env_seen'].get(env_var))!r}")
        if row["note"]:
            print(f"      note: {row['note']}")

In [ ]:
# ── Cross-run invariants ────────────────────────────────────────────────────────
# The marker is normalised, so these pairs must agree key for key. Comparing the
# whole config also covers keys no scenario carries.
def diff_configs(left, right):
    return {key: (left.get(key), right.get(key))
            for key in set(left) | set(right) if left.get(key) != right.get(key)}


checks = []
for a, b, why in [
    ("portal", "portal_mixedcase",
     "'  PoRtAl  ' must be stripped and lowercased to the portal marker"),
    ("manual", "other_marker",
     "an unrecognised marker must behave exactly like no marker"),
]:
    delta = diff_configs(PAYLOADS[a]["config"], PAYLOADS[b]["config"])
    checks.append((not delta, f"{a} == {b}", why, delta))

# Portal and manual must differ, or the gate is not doing anything.
gate_delta = diff_configs(PAYLOADS["portal"]["config"], PAYLOADS["manual"]["config"])
gate_expected = {CONFIG_KEY[s.key] for s in SCENARIOS
                 if expected(s, True) != expected(s, False)}
checks.append((set(gate_delta) == gate_expected, "portal != manual on exactly the gated keys",
               "only the scenarios whose expectations differ may differ",
               {"unexpected": sorted(set(gate_delta) - gate_expected),
                "missing": sorted(gate_expected - set(gate_delta))}))

for ok, name, why, delta in checks:
    print(f"{'PASS' if ok else 'FAIL'}  {name}")
    if not ok:
        print(f"      {why}")
        print(f"      {json.dumps(delta, indent=6, default=str)}")

invariants_failed = [name for ok, name, _, _ in checks if not ok]

---
## Step 7b — Portal vs manual, side by side

Every key `get_config` returns, as the `portal` run resolved it next to the `manual` run.

- **`DIFFERS`** — a scenario whose expectations differ by design. Only the `nx1_` tier is
  portal-gated, so these are the keys where a `nx1_<key>` Variable is the winning tier for the
  portal and invisible to the manual run.
- **`UNEXPECTED`** — differs but no scenario says it should. This is the row to care about: two
  runs of the same DAG, same Variables, differing only in `conf.triggered_by`, should agree
  everywhere else.
- **`same`** — includes every key no scenario carries, which is the evidence the marker changes
  nothing beyond the `nx1_` tier.

### Comparing on real tenant values, without touching anything

The matrix deliberately overwrites Variables to force each tier. To see what a portal run and a
hand-launched run would resolve from the tenant's **actual** `nx1_s3_*` values, trigger the
probe twice with no overlay and no test Variables — read-only, nothing to restore:

```python
for label, conf in [("cmp_portal", {"triggered_by": "portal"}), ("cmp_manual", {})]:
    trigger(run_id_for(label), conf)
wait_for([run_id_for("cmp_portal"), run_id_for("cmp_manual")])
delta = diff_configs(fetch_payload(run_id_for("cmp_portal"))["config"],
                     fetch_payload(run_id_for("cmp_manual"))["config"])
print(json.dumps(delta, indent=2, default=str))   # {key: (portal, manual)}
```

Run that **before** Step 5 or **after** Step 9, otherwise the matrix Variables are still in
place. Each differing key is one the portal is currently overriding for every run it triggers —
if that list is empty, no `nx1_` globals are set on this tenant yet.

### Telling the two apart in Airflow itself

A real portal-triggered run is identifiable three ways, all visible in the UI's run details:
`conf.triggered_by == 'portal'`, a `dag_run_id` of `data_migration_<uuid>`, and a run note
carrying the portal username. A run launched from the UI or the CLI has none of them, which is
exactly what keeps it out of the `nx1_` namespace.

In [ ]:
gated = {CONFIG_KEY[s.key] for s in SCENARIOS if expected(s, True) != expected(s, False)}
carrier_of = {CONFIG_KEY[s.key]: s.sid for s in SCENARIOS}

portal_cfg, manual_cfg = PAYLOADS["portal"]["config"], PAYLOADS["manual"]["config"]

head = (f"{'':<11} {'config key':<26} {'portal run':<36} {'manual run':<36} scenario")
print(head)
print("-" * (len(head) + 12))
for key in sorted(set(portal_cfg) | set(manual_cfg)):
    left, right = portal_cfg.get(key), manual_cfg.get(key)
    if left == right:
        flag = "same"
    else:
        flag = "DIFFERS" if key in gated else "UNEXPECTED"
    print(f"{flag:<11} {key:<26} {render(left):<36} {render(right):<36} "
          f"{carrier_of.get(key, '')}")

surprises = [key for key in set(portal_cfg) | set(manual_cfg)
             if portal_cfg.get(key) != manual_cfg.get(key) and key not in gated]
print(f"\n{len(gated)} keys differ by design — the nx1_ tier is portal-only:")
for key in sorted(gated):
    print(f"    {key:<26} (scenario {carrier_of[key]})")
print(f"\n{'PASS' if not surprises else 'FAIL'}  nothing else differs between the two runs")
for key in sorted(surprises):
    print(f"    UNEXPECTED {key}: portal={render(portal_cfg.get(key))!r} "
          f"manual={render(manual_cfg.get(key))!r}")

---
## Step 8 — The owner fallback chain

`get_config` resolves the owner as

```
_var('migration_dag_owner', ...) or conf['dag_owner'] or conf['spark_user'] or 'data-migration'
```

so the conf fallback is only reachable when every tier is empty. The API writes an **unscoped**
`migration_dag_owner` on every portal run (`_set_airflow_variables`, last line), which lands at
tier 3 and is visible to hand-launched DAGs too — the cell reports whether this tenant has one.

In [ ]:
existing_plain_owner = SANDBOX.saved.get("migration_dag_owner")
print(f"Plain migration_dag_owner on this tenant before the test: {existing_plain_owner!r}")
if existing_plain_owner:
    print("  ^ written unscoped by every portal run; hand-launched DAGs inherit it at tier 3.")

# Every owner tier empty, so the conf fallback is the only thing left.
for key in ("migration_dag_owner", "nx1_migration_dag_owner"):
    SANDBOX.delete(key)

owner_run_id = run_id_for("owner_fallback")
trigger(owner_run_id, {
    "triggered_by": "portal",
    "dag_owner": "probe-conf-owner",
    "reveal_secrets": REVEAL_SECRETS,
    "env_overlay": {"MIGRATION_DAG_OWNER": None},
})
print(f"\nTriggered {owner_run_id}")
wait_for([owner_run_id])

owner_payload = fetch_payload(owner_run_id)
owner_actual = owner_payload["config"]["owner"]
owner_ok = owner_actual == "probe-conf-owner"
print(f"\n{'PASS' if owner_ok else 'FAIL'}  conf['dag_owner'] is used when every tier is empty")
print(f"      expected 'probe-conf-owner', got {owner_actual!r}")

---
## Step 9 — Cleanup and leak check

Restores every borrowed Variable, deletes the run-scoped ones, then repeats the baseline run
and diffs it against Step 4. An identical baseline is the evidence that the probe left the
tenant's resolution exactly as it found it.

In [ ]:
restored, removed, failed_restore = SANDBOX.restore()
print(f"Restored {len(restored)} Variables, deleted {len(removed)} the notebook created")
if failed_restore:
    print("\n  RESTORE FAILURES — fix these by hand:")
    for key, error in failed_restore:
        print(f"    {key}: {error}")

scoped_left = []
for scoped in SCOPED_KEYS_WRITTEN:
    var_delete(scoped)
    if var_get(scoped) is not None:
        scoped_left.append(scoped)
print(f"Deleted {len(SCOPED_KEYS_WRITTEN)} run-scoped Variables"
      + (f" — {len(scoped_left)} still present: {scoped_left}" if scoped_left else ""))

leaked = []
for scenario in SCENARIOS:
    for key in (f"nx1_{scenario.key}", scenario.key):
        current, expected_value = var_get(key), SANDBOX.saved.get(key)
        if key in SANDBOX.saved and current != expected_value:
            leaked.append((key, expected_value, current))
print(f"\n{'PASS' if not leaked else 'FAIL'}  every borrowed Variable is back to its prior value")
for key, was, now in leaked:
    print(f"      {key}: was {was!r}, now {now!r}")

In [ ]:
baseline_post_id = run_id_for("baseline_post")
trigger(baseline_post_id, BASELINE_CONF)
print(f"Triggered {baseline_post_id}")
wait_for([baseline_post_id])
BASELINE_POST = fetch_payload(baseline_post_id)

baseline_delta = diff_configs(BASELINE_PRE["config"], BASELINE_POST["config"])
print(f"\n{'PASS' if not baseline_delta else 'FAIL'}  baseline resolution unchanged by the test")
for key, (before, after) in sorted(baseline_delta.items()):
    print(f"      {key}: {render(before)!r} -> {render(after)!r}")
if baseline_delta:
    print("\n      Causes worth ruling out before treating this as a leak:")
    print("        - a real portal run finished mid-test and rewrote migration_dag_owner")
    print("        - someone edited Variables in the Airflow UI while this ran")
    print("        - a restore failure listed above")

md5_changed = (BASELINE_PRE["diagnostics"]["shared_py_md5"]
               != BASELINE_POST["diagnostics"]["shared_py_md5"])
if md5_changed:
    print("\n      NOTE: deployed shared.py changed during the test — the API re-uploads it "
          "on every startup, so a pod restart can swap the copy under you.")

In [ ]:
# ── Verdict ─────────────────────────────────────────────────────────────────────
summary = [
    (f"scenario matrix ({len(results) - len(failed)}/{len(results)})", not failed),
    ("cross-run invariants", not invariants_failed),
    ("portal vs manual differ only where designed", not surprises),
    ("owner fallback chain", owner_ok),
    ("borrowed Variables restored", not leaked and not failed_restore),
    ("run-scoped Variables cleaned up", not scoped_left),
    ("baseline resolution unchanged", not baseline_delta),
]
print("=" * 60)
for name, ok in summary:
    print(f"  {'PASS' if ok else 'FAIL'}  {name}")
print("=" * 60)

assert all(ok for _, ok in summary), \
    "variable-precedence test failed — see the sections above"
print("\nAll checks passed.")

---
## Step 10 — Remove the probe DAG

Deletes the DAG file first, then the Airflow record — in the other order the bundle sync
re-registers it from the file that is still there.

Leave this until last if you want to inspect task logs in the Airflow UI; deleting the DAG
removes its run history.

In [ ]:
print("Deleted DAG file:" if s3_delete(PROBE_S3_URI) else "DAG file already gone:", PROBE_S3_URI)

af("DELETE", f"/dags/{PROBE_DAG_ID}", allow=(404, 409))
print(f"Deleted Airflow DAG record: {PROBE_DAG_ID}")

purged, failed_purge = purge_probe_xcoms()
print(f"Deleted {len(purged)} probe DAG runs and their XComs"
      + (f" — {len(failed_purge)} failed: {failed_purge}" if failed_purge else ""))
if REVEAL_SECRETS:
    print("  REVEAL_SECRETS was on, so those XComs held plaintext credentials. The "
          "task logs never did. Clear this notebook's output before committing it.")

if os.path.exists(VARIABLE_BACKUP_PATH):
    os.remove(VARIABLE_BACKUP_PATH)
    print(f"Removed the Variable backup (it held real secret values): {VARIABLE_BACKUP_PATH}")

---
## Emergency restore

Run this if the kernel died between Step 5 and Step 9 and the tenant is left holding
`probe-*` values. It replays `~/nx1_var_probe_variable_backup.json`: keys with a recorded
value are written back, keys recorded as `null` did not exist and are deleted.

Also drops any `*__nx1_var_probe_*` run-scoped Variables left behind.

In [ ]:
if not os.path.exists(VARIABLE_BACKUP_PATH):
    print(f"No backup at {VARIABLE_BACKUP_PATH} — nothing to restore.")
else:
    with open(VARIABLE_BACKUP_PATH) as handle:
        saved = json.load(handle)
    for key, value in sorted(saved.items()):
        try:
            if value is None:
                var_delete(key)
                print(f"  deleted (did not exist before): {key}")
            else:
                var_set(key, value)
                print(f"  restored: {key}")
        except Exception as exc:
            print(f"  FAILED {key}: {exc}")

    stale = [entry["key"] for entry in
             (af("GET", "/variables", params={"limit": 1000}) or {}).get("variables", [])
             if f"__{PROBE_DAG_ID}_" in entry.get("key", "")]
    for key in stale:
        var_delete(key)
        print(f"  deleted run-scoped leftover: {key}")

    os.remove(VARIABLE_BACKUP_PATH)
    print(f"\nRestore complete; removed {VARIABLE_BACKUP_PATH}")

---
## Not covered here

- **The portal side.** Which Variables the API writes, `INHERITABLE_RUN_KEYS` skipping blank S3
  fields, and the chained-iceberg re-scope in `DAG2_SCOPED_KEYS` are all in the API, not the
  DAG. Unit tests in `api/tests/` cover those; this notebook starts from Variables already in
  place.
- **Per-endpoint multi-tenant credentials** (`<endpoint-host>_access_key`), read by
  `build_s3_opts` when an Excel row carries an `endpoint`. Resolved outside `_var`.
- **Whether the resolved values work.** This proves the right value is *selected*; it does not
  run distcp against S3. The e2e notebooks cover that.
- **`configure_spark_s3` / `apply_bucket_credentials`**, the only consumers of the retired
  `s3_source_*` / `s3_dest_*` config keys, are not called by any DAG in either repo.